# GeoMag ML — Kaggle T4 Runner

Thin wrapper over `ml_research/scripts/train.py` (no duplicated model or training code).

**Before running:**
1. Upload `ml_research_cloud_bundle.zip` (generated via `ml_research/scripts/pack_for_cloud.py`) as a private Kaggle Dataset and attach it to this notebook.
2. Enable **Accelerator -> GPU T4 x2** (uses `cuda:0`).

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import zipfile

matches = list(Path('/kaggle/input').rglob('ml_research_cloud_bundle.zip'))
if not matches:
    raise FileNotFoundError('Attach the Kaggle Dataset containing ml_research_cloud_bundle.zip.')
BUNDLE_ZIP = matches[0]
WORK_DIR = Path('/kaggle/working/ml_work')
OUT_DIR = Path('/kaggle/working/artifacts_out')
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(BUNDLE_ZIP, 'r') as zf:
    zf.extractall(WORK_DIR)

%cd /kaggle/working/ml_work
!pip install -q -e . --no-deps
try:
    import yaml, pyarrow
except ImportError:
    !pip install -q -r requirements_colab_gpu.txt

In [ ]:
%cd /kaggle/working/ml_work
!pytest -c pytest.ini tests -k "not bridge and not dataset_contract"

In [ ]:
%cd /kaggle/working/ml_work
DATASET = 'data_cache/geomag_ml_oct2026_clean_v1.npz'

# Phase 3 Branch A: Perceiver-Bottleneck AE (4-fold base + key ablations)
!python scripts/train.py --config configs/phase3_branch_a_perceiver.yaml --dataset {DATASET} --fold all --ablation base,k3,k4,no_ut,beta_vae --device cuda --out-dir /kaggle/working/artifacts_out

# Phase 3 Branch B: Cross-Attention Virtual Magnetometer (4-fold base + Zero-Shot + spatial bias ablation)
!python scripts/train.py --config configs/phase3_branch_b_cross_attn.yaml --dataset {DATASET} --fold all --ablation base,zero_shot,no_spatial_bias --device cuda --out-dir /kaggle/working/artifacts_out

In [ ]:
import hashlib
from pathlib import Path
import shutil

OUT_DIR = Path('/kaggle/working/artifacts_out')
ARCHIVE_PATH = Path(shutil.make_archive('/kaggle/working/artifacts_p3_kaggle_t4', 'zip', root_dir=OUT_DIR))
h = hashlib.sha256()
with ARCHIVE_PATH.open('rb') as fh:
    for chunk in iter(lambda: fh.read(1024 * 1024), b''):
        h.update(chunk)
ARCHIVE_PATH.with_suffix('.zip.sha256').write_text(f"{h.hexdigest()}  {ARCHIVE_PATH.name}\n", encoding='utf-8')
print('Saved:', ARCHIVE_PATH, f'({ARCHIVE_PATH.stat().st_size / (1024**2):.2f} MB)')
print('SHA256:', h.hexdigest())